# Data Exploration

Understand the data and validate assumptions before modelling.

- summary statistics  
- target behaviour  
- autocorrelation / random walk checks  
- feature distributions  
- feature vs target relationships  

Goal: identify predictive structure and validate assumptions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from microstructure_alpha.data.loader import load_parquet_glob
from microstructure_alpha.visualization.eda import plot_feature_overview

In [ ]:
final_dataset = load_parquet_glob(
    "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\ml_ready_data\\ml_ready.parquet",
    sort_by="timestamp",
)

# Looking at prediction target (mid_price_change_1)

In [ ]:
def plot_target_overview(df, target, price):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))

    df[target].plot(ax=ax[0])
    df[target].hist(ax=ax[1], bins=100)
    df[price].plot(ax=ax[2])

    ax[0].set_title(target)
    ax[1].set_title(f"{target} distribution")
    ax[2].set_title(price)

    plt.tight_layout()
    plt.show()


plot_target_overview(final_dataset, "mid_price_change_1", "mid_price")

In [ ]:
move_rate = (final_dataset["mid_price_change_1"] != 0).mean()
move_rate

## Target Exploration

The mid_price_change_1 is centred around zero with a high proportion of zero returns, indicating that the price often does not move

The distribution shows heavy tails relative to a Gaussian, with occasional large moves corresponding to periods of increased volatility.

In [ ]:
window = 500

fig, ax = plt.subplots(figsize=(12, 4))

series_dict = {
    "vol": final_dataset["log_return_1"].rolling(window).std() * 1e5,
    "rolling_mean": final_dataset["mid_price_change_1"].rolling(window).mean(),
    "rolling_std": final_dataset["mid_price_change_1"].rolling(window).std(),
}

for label, s in series_dict.items():
    s.plot(ax=ax, label=label)

ax.set_title(f"Rolling stats (window={window})")
ax.legend()
plt.show()

- The rolling mean remains close to zero over time, indicating no persistent drift in returns.

- The rolling standard deviation shows clear periods of elevated volatility, suggesting volatility clustering rather than constant variance.

- Volatility increases significantly in certain intervals, which may correspond to different market regimes.

In [ ]:
plot_feature_overview(final_dataset, ["mid_price_change_1"], figsize=(14, 5), show=True)

In [ ]:
plot_feature_overview(
    final_dataset, ["mid_price_change_1_sign"], figsize=(14, 5), show=True
)

Price changes and log returns show almost no autocorrelation, meaning past moves do not strongly predict future ones. 

However, the sign of the price change shows some short-lag autocorrelation, suggesting smals runs where moves in the same direction can occur for a few steps.

# Checking Random Walk Assumption

- Check if past prices help predict future moves  
- Avoid fitting noise that won’t generalise  
- Understand how efficient the market is at this timescale  
- Motivate need for richer features (order book, flow, etc.)  
- Ensure any later signal is real, not random

In [ ]:
import numpy as np

price = final_dataset["mid_price"].dropna()

hs = np.array([1, 2, 5, 10, 20, 50, 100, 500, 1000, 1500])
vars_ = [price.diff(h).var() for h in hs]
plt.figure(figsize=(6, 4))
plt.plot(hs, vars_, marker="o")
# plt.xscale("log")
plt.xlabel("Horizon")
plt.ylabel("Return variance")
plt.title("Variance vs Return Horizon")

plt.show()
coef = np.polyfit(np.log(hs), np.log(vars_), 1)[0]
print("slope ~", coef)

# Class Imbalance

The data is imbalanced, so the model may over-focus on the majority class.

Even though there are more down moves than up moves, the price still trends upward.

This means:
- Up moves are larger
- Down moves are smaller but more frequent

So price increases are driven by a few large upward moves, not many small ones.

Implication:  
Predicting direction alone is not enough — move size matters.

In [ ]:
final_dataset["mid_price_change_1_sign"].value_counts(normalize=True)

In [ ]:
final_dataset.groupby("mid_price_change_1_sign")["mid_price_change_1"].mean()

In [ ]:
final_dataset["imbalance_1"].mean()

The average imbalance is close to zero.

This means there is no consistent bias in the order book.

So the price increase is likely driven by short periods of strong imbalance, not a constant effect.

# Feature Analysis

Now we analyse the features to see if they contain useful predictive signal.

In [ ]:
from microstructure_alpha.features.feature_lists import FEATURE_GROUPS

In [ ]:
from pathlib import Path

base_path = Path(
    "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\notebooks\\notebook_figs\\03_figs"
)

for name, cols in FEATURE_GROUPS.items():
    print(name, cols)

    save_dir = base_path / name
    save_dir.mkdir(parents=True, exist_ok=True)

    plot_feature_overview(
        final_dataset,
        cols,
        figsize=(14, 5),
        show=False,
        save_path=save_dir,
    )

### Feature Summary

**Spread**
- Very small and stable (often at tick size)
- Occasional spikes during high volatility
- No autocorrelation

**Order Book Imbalance**
- Rapidly shifts between buy/sell pressure
- Slight negative mean
- Short-term autocorrelation

**Liquidity**
- Mostly stable with occasional spikes
- Right-skewed distributions (rare large bursts)
- Ratios can produce extreme outliers

**Microprice**
- Tracks price smoothly
- Usually close to mid price
- Spikes align with volatility changes
- No autocorrelation in changes

**Returns & Momentum**
- Centred around zero with small values
- Heavy tails from occasional large moves
- Apparent autocorrelation from overlapping windows

**Volatility**
- Low most of the time with clustered spikes
- Clear autocorrelation (volatility clustering)

**Trade Activity & Volume**
- Irregular bursts of activity
- Heavy-tailed distributions
- Large trades drive spikes more than many small ones

**Trade Flow**
- Covers full range [-1, 1], often extreme

**Trade Dynamics & Lags**
- Changes mostly small with occasional spikes
- Lagged features mirror original behaviour